In [12]:
%pylab inline
%config InlineBackend.figure_format = 'retina'
from ipywidgets import interact
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
tf.random.set_seed(42)
np.random.seed(42)

from Training_Data.Particle_Tracking_Training_Data import Particle_Tracking_Training_Data
from tensorflow.keras.utils import register_keras_serializable
from tensorflow.keras import layers, models
from models.particle_model import particle_tracking_model, compile_particle_model
from models.particle_model import load_particle_model

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


# Procedurally generated training data
The code below demonstrates how to generate training videos and labels. The function also returns the ground truth particle tracks, which might also be useful depending on your goals.

Note that the training generator is a Tensorflow Module and can be easily incorperated into a Tensorflow neural network. Alternatively, you could simply save a large set of data and use another machine learning framework.

Note that the image dimension is fixed at 256x256. The labels are downsampled to 128x128 in the image dimensions. There are two classes (a particle is detected or not detected) per label so the label shape is 128x128x2. Hence, the neural network output should be 128x128x2.

In [2]:
Nt = 50 ## number of frames for each video
kappa = 0.1 ## standard deviation of background noise added to image
a = 3. ## scale factor for the size of particle spots (not true size of particles)
IbackLevel = 0.15 ## relative intensity of randomly generated background pattern; in (0, 1)
Nparticles = 10 ## the number of particles (more => slower)
sigma_motion = 2.3 ## the standard deviation for particle brownian motion; should be in (0, 10)

## you might consider randomizing some of these parameters when training a neural net

pt = Particle_Tracking_Training_Data(Nt) ## create object instance
## you can 'call' the object as many times as you want
## in this example, we only generate one training example
vid, labels, tracks = pt(kappa, a, IbackLevel, Nparticles, sigma_motion) 

## Visualizing training videos and labels

In [3]:
@interact(t=(0, Nt-1, 1))
def plotfn(t=0, show_tracks=True):
    fig = figure(1, [14, 7])
    fig.add_subplot(121)
    imshow(vid[t], origin='lower')
    if show_tracks:
        plot(tracks[t, :, 0], tracks[t, :, 1], 'rx')
    xlim(-10, 265)
    ylim(-10, 265)
    
    fig.add_subplot(122)
    imshow(vid[t], origin='lower')
    # labels are 128x128; align overlay to the 256x256 image canvas
    imshow(labels[t, ..., 1], origin='lower', cmap='spring', alpha=0.5, interpolation='nearest', extent=(0, 256, 0, 256))

interactive(children=(IntSlider(value=0, description='t', max=49), Checkbox(value=True, description='show_trac…

# Design and train a convolutional neural network using the training data generator

In [ ]:
model = particle_tracking_model()
compile_particle_model(model)

In [ ]:
# Train the model (example)
# X = vid[..., None] if vid.ndim == 3 else vid  # (Nt, 256, 256, 1)
# history = model.fit(X, labels, epochs=10, validation_split=0.2, batch_size=4)

In [ ]:
# # Use more videos to train the model (example)
# num_videos = 10
# vids = []
# lbls = []
# for i in range(num_videos):
#     v, l, _ = pt(kappa, a, IbackLevel, Nparticles, sigma_motion)
#     vids.append(v[..., None])  # add channel dimension
#     lbls.append(l)
# X = np.concatenate(vids, axis=0)  # (num_frames_total, 256, 256, 1)
# Y = np.concatenate(lbls, axis=0)  # (num_frames_total, 128, 128, 2)
# model.fit(X, Y, batch_size=8, epochs=15, validation_split=0.2)

In [ ]:
# Save model (if using a custom loss, provide custom_objects when loading)
# model.save('model_new_6.keras')

In [13]:
model = load_particle_model('model_new.keras')

In [8]:
# Test samples
test_vid, test_labels, test_tracks = pt(kappa, a, IbackLevel, Nparticles, sigma_motion)

# Visualizing testing videos and labels
@interact(t=(0, Nt-1, 1))
def plotfn(t=0, show_tracks=True):
    fig = figure(1, [14, 7])
    fig.add_subplot(121)
    imshow(test_vid[t], origin='lower')
    if show_tracks:
        plot(test_tracks[t, :, 0], test_tracks[t, :, 1], 'rx')
    xlim(-10, 265)
    ylim(-10, 265)
    
    fig.add_subplot(122)
    imshow(test_vid[t], origin='lower')
    imshow(test_labels[t, ..., 1], origin='lower')

interactive(children=(IntSlider(value=0, description='t', max=49), Checkbox(value=True, description='show_trac…

In [ ]:
# Use our model (ensure channel dimension)
if test_vid.ndim == 3:
    test_vid_in = test_vid[..., None]
else:
    test_vid_in = test_vid

predictions = model.predict(test_vid_in)

# Take the 2nd channel as particle probability; shape: [Nt, H, W]
pred_probs = predictions[..., 1]

# Interactive visualization of predictions
@interact(t=(0, Nt-1, 1), threshold=(0.1, 0.9, 0.05), show_tracks=True, show_binary=False)
def show_prediction(t=0, threshold=0.5, show_tracks=True, show_binary=False):
    fig = figure(1, [14, 7])

    # Left: raw frame + tracks
    fig.add_subplot(121)
    imshow(test_vid[t], origin='lower')
    if show_tracks:
        plot(test_tracks[t, :, 0], test_tracks[t, :, 1], 'rx')
    xlim(-10, 265)
    ylim(-10, 265)

    # Right: raw frame + prediction (heatmap or binary mask)
    fig.add_subplot(122)
    imshow(test_vid[t], origin='lower')
    if show_binary:
        pred_mask = pred_probs[t] > threshold
        imshow(pred_mask, origin='lower', cmap='spring', alpha=0.5, interpolation='nearest', extent=(0, 256, 0, 256))
        title(f"Binary mask @ thr={threshold:.2f}")
    else:
        imshow(pred_probs[t], origin='lower', cmap='magma', alpha=0.5, interpolation='nearest', extent=(0, 256, 0, 256))
        title("Probability heatmap")

2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 394ms/step
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 394ms/step


interactive(children=(IntSlider(value=0, description='t', max=49), FloatSlider(value=0.5, description='thresho…

In [ ]:
# (Placeholder to keep cell order tidy; you can remove this cell)